In [4]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque

# Initialize deque for points of different colors
bpoints = [deque(maxlen=1024)]
gpoints = [deque(maxlen=1024)]
rpoints = [deque(maxlen=1024)]
ypoints = [deque(maxlen=1024)]

# Indexes to mark points in the array of specific colors
blue_index = 0
green_index = 0
red_index = 0
yellow_index = 0

# Kernel for dilation
kernel = np.ones((5, 5), np.uint8)

# Colors: Blue, Green, Red, Yellow
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255)]
colorIndex = 0

# Create a white canvas
paintWindow = np.zeros((471, 636, 3)) + 255
paintWindow = cv2.rectangle(paintWindow, (40, 1), (140, 65), (0, 0, 0), 2)
paintWindow = cv2.rectangle(paintWindow, (160, 1), (255, 65), (255, 0, 0), 2)
paintWindow = cv2.rectangle(paintWindow, (275, 1), (370, 65), (0, 255, 0), 2)
paintWindow = cv2.rectangle(paintWindow, (390, 1), (485, 65), (0, 0, 255), 2)
paintWindow = cv2.rectangle(paintWindow, (505, 1), (600, 65), (0, 255, 255), 2)

# Add labels to the buttons
cv2.putText(paintWindow, "CLEAR", (49, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
cv2.putText(paintWindow, "BLUE", (185, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
cv2.putText(paintWindow, "GREEN", (298, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
cv2.putText(paintWindow, "RED", (420, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
cv2.putText(paintWindow, "YELLOW", (520, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

cv2.namedWindow("White Board", cv2.WINDOW_AUTOSIZE)

# Initialize Mediapipe hands solution
mpHands = mp.solutions.hands
hands = mpHands.Hands(max_num_hands=1, min_detection_confidence=0.7)

# Initialize drawing utils
mpDraw = mp.solutions.drawing_utils

# Initialize webcam
cap = cv2.VideoCapture(0)

while True:
    # Read each frame from the webcam
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Get hand landmark prediction
    result = hands.process(frame_rgb)

    # Display color buttons on the screen
    frame = cv2.rectangle(frame, (40, 1), (140, 65), (0, 0, 0), 2)
    frame = cv2.rectangle(frame, (160, 1), (255, 65), (255, 0, 0), 2)
    frame = cv2.rectangle(frame, (275, 1), (370, 65), (0, 255, 0), 2)
    frame = cv2.rectangle(frame, (390, 1), (485, 65), (0, 0, 255), 2)
    frame = cv2.rectangle(frame, (505, 1), (600, 65), (0, 255, 255), 2)

    cv2.putText(frame, "CLEAR", (49, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, "BLUE", (185, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, "GREEN", (298, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, "RED", (420, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, "YELLOW", (520, 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2, cv2.LINE_AA)

    # Post-process the result
    if result.multi_hand_landmarks:
        landmarks = []
        for handLms in result.multi_hand_landmarks:
            for lm in handLms.landmark:
                lmx = int(lm.x * frame.shape[1])
                lmy = int(lm.y * frame.shape[0])
                landmarks.append((lmx, lmy))

            # Draw landmarks on frame
            mpDraw.draw_landmarks(frame, handLms, mpHands.HAND_CONNECTIONS)

        # Calculate coordinates for the forefinger and thumb
        fore_finger = (landmarks[8][0], landmarks[8][1])
        thumb = (landmarks[4][0], landmarks[4][1])
        cv2.circle(frame, fore_finger, 3, (0, 255, 0), -1)

        # Check if forefinger and thumb are close (new stroke)
        if abs(thumb[1] - fore_finger[1]) < 30:
            bpoints.append(deque(maxlen=512))
            blue_index += 1
            gpoints.append(deque(maxlen=512))
            green_index += 1
            rpoints.append(deque(maxlen=512))
            red_index += 1
            ypoints.append(deque(maxlen=512))
            yellow_index += 1

        # Check if the forefinger is selecting a button
        elif fore_finger[1] <= 65:
            if 40 <= fore_finger[0] <= 140:  # Clear button
                bpoints = [deque(maxlen=512)]
                gpoints = [deque(maxlen=512)]
                rpoints = [deque(maxlen=512)]
                ypoints = [deque(maxlen=512)]
                blue_index = green_index = red_index = yellow_index = 0
                paintWindow[67:, :, :] = 255
            elif 160 <= fore_finger[0] <= 255:  # Blue color
                colorIndex = 0
            elif 275 <= fore_finger[0] <= 370:  # Green color
                colorIndex = 1
            elif 390 <= fore_finger[0] <= 485:  # Red color
                colorIndex = 2
            elif 505 <= fore_finger[0] <= 600:  # Yellow color
                colorIndex = 3
        else:
            if colorIndex == 0:
                bpoints[blue_index].appendleft(fore_finger)
            elif colorIndex == 1:
                gpoints[green_index].appendleft(fore_finger)
            elif colorIndex == 2:
                rpoints[red_index].appendleft(fore_finger)
            elif colorIndex == 3:
                ypoints[yellow_index].appendleft(fore_finger)

    # Draw lines of all the colors on the canvas and frame
    points = [bpoints, gpoints, rpoints, ypoints]
    for i in range(len(points)):
        for j in range(len(points[i])):
            for k in range(1, len(points[i][j])):
                if points[i][j][k - 1] is None or points[i][j][k] is None:
                    continue
                cv2.line(frame, points[i][j][k - 1], points[i][j][k], colors[i], 2)
                cv2.line(paintWindow, points[i][j][k - 1], points[i][j][k], colors[i], 2)

    cv2.imshow("outshow", frame)
    cv2.imshow("White Board", paintWindow)

    if cv2.waitKey(1) & 0xFF == 27:  # Press 'Esc' to exit
        break

cap.release()
cv2.destroyAllWindows()
hands.close()

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import csv
import os

# Initialize Mediapipe Hands
mpHands = mp.solutions.hands
hands = mpHands.Hands(max_num_hands=1, min_detection_confidence=0.7)
mpDraw = mp.solutions.drawing_utils

# Initialize webcam
cap = cv2.VideoCapture(0)

# CSV File Setup
csv_filename = 'gesture_data.csv'
if not os.path.exists(csv_filename):
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Define descriptive column names for the CSV file
        writer.writerow([
            'Gesture_Label',  # The label assigned to the gesture (e.g., 0, 1, 2, etc.)
            'Index_Finger_X',  # X-coordinate of the index finger tip
            'Index_Finger_Y',  # Y-coordinate of the index finger tip
            'Thumb_X',         # X-coordinate of the thumb tip
            'Thumb_Y'          # Y-coordinate of the thumb tip
        ])

# Function to save data to CSV
def save_to_csv(label, fore_finger, thumb):
    with open(csv_filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([label, fore_finger[0], fore_finger[1], thumb[0], thumb[1]])

# Variables to control label input
current_label = None
collecting_data = False

print("Press a number key (0-9) to set the label and start collecting data.")
print("Press 'c' to stop collecting data.")
print("Press 'Esc' to exit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(frame_rgb)

    if result.multi_hand_landmarks:
        landmarks = []
        for handLms in result.multi_hand_landmarks:
            mpDraw.draw_landmarks(frame, handLms, mpHands.HAND_CONNECTIONS)
            for lm in handLms.landmark:
                lmx = int(lm.x * frame.shape[1])
                lmy = int(lm.y * frame.shape[0])
                landmarks.append((lmx, lmy))

        # Get positions of forefinger and thumb
        fore_finger = (landmarks[8][0], landmarks[8][1])  # Index finger tip
        thumb = (landmarks[4][0], landmarks[4][1])        # Thumb tip

        # Draw circles on forefinger and thumb for visualization
        cv2.circle(frame, fore_finger, 5, (0, 255, 0), -1)
        cv2.circle(frame, thumb, 5, (255, 0, 0), -1)

        # If collecting data, save the coordinates to the CSV file
        if collecting_data and current_label is not None:
            save_to_csv(current_label, fore_finger, thumb)

    # Display instructions on the frame
    cv2.putText(frame, "Set Label (0-9), 'c' to stop, 'Esc' to exit", 
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Data Collection", frame)

    # Handle key presses
    key = cv2.waitKey(1) & 0xFF

    if key == 27:  # ESC to exit
        break
    elif key >= ord('0') and key <= ord('9'):  # Set label (0-9)
        current_label = chr(key)
        collecting_data = True
        print(f"Collecting data for label: {current_label}")
    elif key == ord('c'):  # Stop collecting data
        collecting_data = False
        print("Stopped collecting data.")

cap.release()
cv2.destroyAllWindows()
hands.close()

Press a number key (0-9) to set the label and start collecting data.
Press 'c' to stop collecting data.
Press 'Esc' to exit.
